<a href="https://colab.research.google.com/github/wantuds-crypto/Ciencia-datos-curso/blob/main/Final_precios_inmuebles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Introducción y objetivo del análisis
### El problema
**Problemática:** predecir o explicar el precio de venta de una vivienda a partir de sus características físicas y de ubicación, para apoyar decisiones como fijar el precio de listado, detectar propiedades sub o sobrevaloradas y entender qué atributos elevan más el valor de una casa.

El precio de una vivienda depende de muchas cosas a la vez: su tamaño, su estado, su antigüedad, si tiene vista o frente al agua y, sobre todo, **dónde está**. Para quien compra, vende, construye o financia vivienda, entender **qué características se asocian con un precio más alto** permite tomar mejores decisiones: fijar precios de lista, priorizar zonas de inversión, decidir si conviene renovar o detectar ventas con precios anómalos.

### Objetivo general
Identificar, con los datos disponibles, qué características de la vivienda y de su ubicación se relacionan más con su **precio de venta**, y dejar el terreno preparado (datos limpios, variables construidas y evaluadas) para un futuro modelo de estimación de precios.

### Preguntas que queremos responder
1. ¿Qué tan confiables son los datos y qué problemas de calidad tienen?
2. ¿Cómo se distribuye el precio de venta? ¿Hay valores imposibles o sospechosos?
3. ¿Qué zonas y ciudades concentran las ventas y cuáles tienen precios más altos?
4. ¿Cómo cambia el precio según el tamaño, la antigüedad, el estado y la calidad de la vista?
5. ¿Qué variables (originales o construidas) tienen la relación más fuerte con el precio?
6. ¿Qué recomendaciones prácticas y qué próximos pasos se derivan?
\

In [213]:
# --- Librerías ---
from pathlib import Path
import re
import warnings
import os
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import duckdb
from scipy import stats
import kagglehub

In [214]:
# Descarga del dataset
path = kagglehub.dataset_download("debayank2024/house-price-prediction")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'house-price-prediction' dataset.
Path to dataset files: /kaggle/input/house-price-prediction


## 1. Primer vistazo a los datos

Antes de limpiar o transformar nada, simplemente abrimos el archivo para ver que contiene: cuantas ventas hay, que columnas trae y como se ven los primeros registros. Este paso no cambia ni corrige nada, solo nos deja ver el punto de partida. El archivo trae 4.600 ventas de viviendas; cada fila es una casa vendida, con datos como precio, numero de habitaciones, tamano, ano de construccion y ubicacion. A partir de aqui decidiremos que hay que corregir.

In [215]:
df = pd.read_csv(os.path.join(path, "modified_data.csv"))
print("Filas y columnas:", df.shape)
df.info()
df.head()

Filas y columnas: (4600, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            4600 non-null   object 
 1   price           4600 non-null   float64
 2   bedrooms        4600 non-null   float64
 3   bathrooms       4600 non-null   float64
 4   sqft_living     4600 non-null   int64  
 5   sqft_lot        4600 non-null   int64  
 6   floors          4600 non-null   float64
 7   waterfront      4600 non-null   int64  
 8   view            4600 non-null   int64  
 9   condition       4600 non-null   int64  
 10  sqft_above      4600 non-null   int64  
 11  sqft_basement   4600 non-null   int64  
 12  yr_built        4600 non-null   int64  
 13  yr_renovated    4600 non-null   int64  
 14  street          4600 non-null   object 
 15  city            4600 non-null   object 
 16  statezip        4600 non-null   object 
 17  pric

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,price_per_sqft
0,2014-05-02,"313,000.00",3.00,1.50,1340,7912,1.50,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,233.58
1,2014-05-02,"2,384,000.00",5.00,2.50,3650,9050,2.00,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,653.15
2,2014-05-02,"342,000.00",3.00,2.00,1930,11947,1.00,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,177.20
3,2014-05-02,"420,000.00",3.00,2.25,2000,8030,1.00,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,210.00
4,2014-05-02,"550,000.00",4.00,2.50,1940,10500,1.00,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,283.51


In [216]:
df.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,price_per_sqft
count,"4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00"
mean,"551,962.99",3.40,2.16,"2,139.35","14,852.52",1.51,0.01,0.24,3.45,"1,827.27",312.08,"1,970.79",808.61,265.88
std,"563,834.70",0.91,0.78,963.21,"35,884.44",0.54,0.08,0.78,0.68,862.17,464.14,29.73,979.41,357.50
min,0.00,0.00,0.00,370.00,638.00,1.00,0.00,0.00,1.00,370.00,0.00,"1,900.00",0.00,0.00
25%,"322,875.00",3.00,1.75,"1,460.00","5,000.75",1.00,0.00,0.00,3.00,"1,190.00",0.00,"1,951.00",0.00,180.82
50%,"460,943.46",3.00,2.25,"1,980.00","7,683.00",1.50,0.00,0.00,3.00,"1,590.00",0.00,"1,976.00",0.00,243.86
75%,"654,962.50",4.00,2.50,"2,620.00","11,001.25",2.00,0.00,0.00,4.00,"2,300.00",610.00,"1,997.00","1,999.00",314.84
max,"26,590,000.00",9.00,8.00,"13,540.00","1,074,218.00",3.50,1.00,4.00,5.00,"9,410.00","4,820.00","2,014.00","2,014.00","22,533.90"


## 2. Limpieza de los datos

 Encontramos varios problemas "escondidos": ceros que no son ceros reales sino la forma en que el sistema marca "no aplica" o "no se registro", y algunos precios que son errores evidentes de digitacion. La regla que seguimos en todo el proceso es la misma: nunca eliminamos una fila completa; en cambio, marcamos el problema con una "bandera" y excluimos esos registros solo de los calculos donde afectarian el resultado. Asi, cualquier persona puede revisar despues que se marco y por que, y decidir si esta de acuerdo. A continuacion se resumen los cuatro grupos de problemas que se corrigieron.

#Formato Fechas
-pasar el date a formato fecha y además agregar una columna solo con el mes para determinar si hay meses que las viviendas son más costosas que otros.

In [217]:
df["date"] = pd.to_datetime(df["date"])
df = df.assign(
    mes=df['date'].dt.month
)

In [218]:
df.select_dtypes(include=['bool']).info()
df.select_dtypes(include=['object']).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Empty DataFrame
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   street    4600 non-null   object
 1   city      4600 non-null   object
 2   statezip  4600 non-null   object
dtypes: object(3)
memory usage: 107.9+ KB


In [219]:
print("Nulos por columna:")
print(df.isnull().sum())

print("\nFilas duplicadas exactas:", df.duplicated().sum())

Nulos por columna:
date              0
price             0
bedrooms          0
bathrooms         0
sqft_living       0
sqft_lot          0
floors            0
waterfront        0
view              0
condition         0
sqft_above        0
sqft_basement     0
yr_built          0
yr_renovated      0
street            0
city              0
statezip          0
price_per_sqft    0
mes               0
dtype: int64

Filas duplicadas exactas: 0


In [244]:
import numpy as np
import re

MAPA_COLUMNAS = {
    "date": "fecha", "price": "precio", "bedrooms": "habitaciones", "bathrooms": "banos",
    "sqft_living": "area_construida_sqft", "sqft_lot": "area_lote_sqft", "floors": "pisos",
    "waterfront": "frente_agua", "view": "vista", "condition": "condicion",
    "sqft_above": "area_sobre_suelo_sqft", "sqft_basement": "area_sotano_sqft",
    "yr_built": "anio_construccion", "yr_renovated": "anio_renovacion", "street": "direccion",
    "city": "ciudad", "statezip": "estado_zip", "price_per_sqft": "precio_por_sqft",
}

def limpiar(raw: pd.DataFrame):
    df = raw.copy()
    bitacora = []

    # 1) Nombres, texto e identificador unico
    df = df.rename(columns=MAPA_COLUMNAS)
    df.columns = [re.sub(r"[^a-z0-9_]", "_", c.strip().lower()) for c in df.columns]
    for c in ["direccion", "ciudad", "estado_zip"]:
        df[c] = df[c].astype("string").str.strip().str.replace(r"\s{2,}", " ", regex=True)
    df["ciudad"] = df["ciudad"].str.title().str.replace("Seatac", "SeaTac")
    partes = df["estado_zip"].str.extract(r"^(?P<estado>[A-Z]{2})\s(?P<codigo_postal>\d{5})$")
    df = pd.concat([df.drop(columns="estado_zip"), partes], axis=1)
    df.insert(0, "id_inmueble", [f"INM-{i:05d}" for i in range(1, len(df) + 1)])
    bitacora.append(["1. Nombres, texto e identificador",
                      "Columnas en ingles sin id unico; texto con espacios/mayusculas inconsistentes",
                      "Traducir nombres, limpiar texto, separar estado/codigo postal, crear id_inmueble",
                      len(df)])

    # 2) Precio en cero y precios imposibles
    df["flag_precio_no_registrado"] = df["precio"] == 0
    df.loc[df["flag_precio_no_registrado"], "precio"] = np.nan
    df["precio_por_sqft"] = (df["precio"] / df["area_construida_sqft"]).round(2)
    log_pps = np.log(df["precio_por_sqft"].where(df["precio_por_sqft"] > 0))
    q1, q3 = log_pps.quantile([.25, .75]); iqr = q3 - q1
    df["flag_precio_extremo"] = ((log_pps < q1 - 3 * iqr) | (log_pps > q3 + 3 * iqr)).fillna(False)
    bitacora.append(["2. Precios en cero y precios imposibles",
                      f"{int(df['flag_precio_no_registrado'].sum())} ventas en US$0 y precios por pie2 absurdos",
                      "Marcar como nulo/atipico y excluir del analisis del precio (sin borrar filas)",
                      int(df["flag_precio_no_registrado"].sum() + df["flag_precio_extremo"].sum())])

    # 3) Habitaciones/banos en cero y renovacion inconsistente
    m0 = (df["habitaciones"] == 0) & (df["banos"] == 0)
    df.loc[m0, ["habitaciones", "banos"]] = np.nan
    m_nunca = df["anio_renovacion"] == 0
    m_inc = (df["anio_renovacion"] > 0) & (df["anio_renovacion"] < df["anio_construccion"])
    df["flag_renovacion_inconsistente"] = m_inc
    df.loc[m_nunca | m_inc, "anio_renovacion"] = np.nan
    bitacora.append(["3. Habitaciones/banos en 0 y renovacion inconsistente",
                      f"{int(m0.sum())} casas grandes con 0 habitaciones/banos; anos de renovacion en 0 o anteriores a la construccion",
                      "Convertir a nulo (dato desconocido / no aplica) y marcar inconsistencias",
                      int(m0.sum() + (m_nunca | m_inc).sum())])

    # 4) Direcciones repetidas y lotes atipicos
    df["flag_direccion_repetida"] = df.duplicated(["direccion", "ciudad", "codigo_postal"], keep=False)
    bitacora.append(["4. Direcciones repetidas y lotes muy grandes",
                      "Direcciones repetidas (casas distintas) y lotes rurales muy grandes",
                      "No se elimina nada: se marcan direcciones repetidas y se recomienda usar medianas para los lotes",
                      int(df["flag_direccion_repetida"].sum())])

    return df, pd.DataFrame(bitacora, columns=["Paso", "Problema encontrado", "Que se hizo", "Registros afectados"])

df, bitacora = limpiar(df_Path)   # usa el nombre que crea la celda de carga
bitacora

NameError: name 'df_Path' is not defined

**1. Nombres, texto e identificador unico.** Las columnas venian sin un identificador propio para cada vivienda, se limpiaron espacios y mayusculas en direccion y ciudad, se separo la columna que mezclaba estado y codigo postal en dos columnas distintas, y se creo un identificador (`id_inmueble`) para cada vivienda, porque la direccion se repite entre casas distintas (por ejemplo, conjuntos de apartamentos) y no sirve como identificador confiable por si sola.

**2. Precios en cero y precios imposibles.** 49 ventas (poco mas del 1%) tenian un precio de 0 dolares, lo cual no es un precio de mercado real; se marcaron y se convirtieron en "dato faltante", conservando la vivienda para el resto del analisis. Ademas, se detectaron 5 ventas con un precio por pie cuadrado absurdo, como una casa de 1.180 pies2 registrada en 26,6 millones de dolares, o ventas de apenas 7.800 y 84.350 dolares. Estos 5 casos se marcaron como "precio extremo" y se excluyen de cualquier analisis relacionado con el precio, aunque la vivienda sigue existiendo en la base de datos.

**3. Habitaciones y banos en cero, y renovaciones inconsistentes.** Dos viviendas grandes aparecian con 0 habitaciones y 0 banos, algo fisicamente poco creible; esos valores se convirtieron en "dato desconocido". Tambien encontramos que la columna de ano de renovacion usa el numero 0 para decir "nunca se ha renovado" (no que se renovo en el ano 0), y que 195 registros tenian una fecha de renovacion anterior a la fecha de construccion, lo cual es imposible. En ambos casos se corrigio el valor a "sin dato" y se dejo una marca para poder rastrear la decision.

**4. Direcciones repetidas y terrenos muy grandes.** Varias viviendas comparten la misma direccion base, pero corresponden a casas distintas (diferente tamano, habitaciones o ano de construccion), asi que no se eliminaron: solo se marcaron como "direccion repetida" para tenerlo presente. De forma similar, algunos terrenos son extremadamente grandes (lotes rurales), lo cual es real y no un error; se conservan, pero se recomienda usar la mediana en vez del promedio al resumir esta variable, porque el promedio se distorsiona con estos valores.

## 3. Casteo: darle a cada dato el tipo correcto
la fecha debe tratarse como fecha (para poder calcular antiguedad), un numero entero como numero entero, y un codigo postal debe guardarse como texto para no perder los ceros a la izquierda. Antes de este paso, por ejemplo, la fecha de venta llegaba como texto plano y no se podia usar para calcular meses o antiguedad. Este paso no cambia los valores, solo la forma en que se guardan.

In [221]:
# Casteo de tipos
df['date'] = pd.to_datetime(df['date'])
df['waterfront'] = df['waterfront'].astype(bool)

for c in ['city', 'street', 'statezip']:
    df[c] = df[c].astype('category')

cat_condition = pd.CategoricalDtype(categories=sorted(df['condition'].unique()), ordered=True)
df['condition'] = df['condition'].astype(cat_condition)

cat_view = pd.CategoricalDtype(categories=sorted(df['view'].unique()), ordered=True)
df['view'] = df['view'].astype(cat_view)

df.dtypes

,0
date,datetime64[ns]
price,float64
bedrooms,float64
bathrooms,float64
sqft_living,int64
sqft_lot,int64
floors,float64
waterfront,bool
view,category
condition,category


In [222]:
# Booleana: 2 estados que representan sí/no
df['waterfront'] = df['waterfront'].astype(bool)

# Nominal: categorías sin orden -> category simple
df['city'] = df['city'].astype('category')

# Ordinal: categorías CON orden -> category con ordered=True
cat_condition = pd.CategoricalDtype(categories=[1,2,3,4,5], ordered=True)
df['condition'] = df['condition'].astype(cat_condition)

###Analsis Inicial:
- Nominales: `street`, `city`, `statezip`
- Ordinales: `condition` , `view` , `floors`
- Booleanas: `waterfront`
- Numéricas(no nominal / ordinal / booleana):`price`, `sqft_living`, `sqft_lot`, `sqft_above`, `sqft_basement`, `price_per_sqft`,`bedrooms`, `bathrooms`, `yr_built`, `yr_renovated`,`yr_renovated`, `date`


In [245]:
CASTING = {
    "id_inmueble": "string", "fecha": "datetime", "precio": "float64",
    "habitaciones": "Int64", "banos": "float64", "area_construida_sqft": "Int64",
    "area_lote_sqft": "Int64", "pisos": "float64", "frente_agua": "boolean",
    "vista": "Int8", "condicion": "Int8", "area_sobre_suelo_sqft": "Int64",
    "area_sotano_sqft": "Int64", "anio_construccion": "Int16", "anio_renovacion": "Int16",
    "direccion": "string", "ciudad": "category", "estado": "category",
    "codigo_postal": "string", "precio_por_sqft": "float64",
}

for col, tipo in CASTING.items():
    df[col] = pd.to_datetime(df[col]) if tipo == "datetime" else df[col].astype(tipo)

df.dtypes.loc[list(CASTING)]

,0
id_inmueble,string[python]
fecha,datetime64[ns]
precio,float64
habitaciones,Int64
banos,float64
area_construida_sqft,Int64
area_lote_sqft,Int64
pisos,float64
frente_agua,boolean
vista,Int8


**Que logramos con esto.** Ahora la fecha es una fecha real (se puede calcular antiguedad o mes de venta), las habitaciones son numeros enteros, el codigo postal se guarda como texto de 5 digitos (sin perder ceros iniciales) y la ciudad se guarda como categoria, lo que ademas ahorra memoria porque son pocas ciudades repetidas muchas veces. Verificamos que ninguna fila se haya perdido en el proceso y que el precio por pie cuadrado siga siendo consistente con el precio y el area construida. Todo quedo listo para el resto del analisis.

In [223]:
# Filas donde price o sqft_living son 0 o negativos
filas_invalidas = df[(df['price'] <= 0) | (df['sqft_living'] <= 0)]
print("Filas inválidas:", len(filas_invalidas))
df = df[(df['price'] > 0) & (df['sqft_living'] > 0)].reset_index(drop=True)

Filas inválidas: 49


In [224]:
# Agrupación geográfica de ciudades en zonas:
ZONAS = {
    "Seattle": ["Seattle"],
    "Eastside": ["Bellevue", "Redmond", "Kirkland", "Sammamish", "Issaquah", "Mercer Island", "Medina",
                 "Clyde Hill", "Yarrow Point", "Newcastle", "Woodinville", "Bothell", "Kenmore",
                 "Beaux Arts Village", "Inglewood-Finn Hill"],
    "Norte": ["Shoreline", "Lake Forest Park"],
    "Sur": ["Renton", "Kent", "Auburn", "Federal Way", "Burien", "Des Moines", "Tukwila", "SeaTac",
            "Normandy Park", "Covington", "Maple Valley", "Algona", "Pacific", "Milton", "Black Diamond", "Enumclaw"],
    "Rural / Cascadas": ["Snoqualmie", "North Bend", "Duvall", "Carnation", "Fall City", "Preston",
                         "Ravensdale", "Skykomish", "Snoqualmie Pass", "Vashon"],
}
CIUDAD_A_ZONA = {c: z for z, ciudades in ZONAS.items() for c in ciudades}


## Creamos llaves para poder hacer un análisis relacional real con JOINs sin inventar informacion.

| Tabla del modelo | | Llave primaria |,
|---|---|---|---|
| `inmuebles` | | `id_inmueble`
| `ventas` | | `id_venta` |
| `dim_ciudades` | | `ciudad` |
| `referencia_zip` | | `codigo_postal`|
              

## 4. Las 22 variables limpias

Despues de limpiar y de asignarle el tipo correcto a cada dato, nos quedamos con una tabla base de 22 variables (columnas). Estas son la version "confiable" de los datos: sin ceros escondidos, sin precios imposibles, y con cada dato guardado en el formato que le corresponde. A partir de esta tabla se construyen las consultas en SQL, los graficos y la evaluacion de variables que siguen. Esta es, literalmente, la base de todo lo demas.


In [246]:
variables_limpias = pd.DataFrame([
    ["id_inmueble", "Identificador unico de cada vivienda"],
    ["fecha", "Dia en que se realizo la venta"],
    ["precio", "Precio de venta en dolares (variable objetivo / Target)"],
    ["habitaciones", "Numero de habitaciones"],
    ["banos", "Numero de banos"],
    ["area_construida_sqft", "Area construida, en pies cuadrados"],
    ["area_lote_sqft", "Area del terreno, en pies cuadrados"],
    ["pisos", "Numero de pisos"],
    ["frente_agua", "Si la vivienda tiene frente al agua (si/no)"],
    ["vista", "Calidad de la vista, de 0 (sin vista) a 4 (excelente)"],
    ["condicion", "Estado de conservacion, de 1 (malo) a 5 (excelente)"],
    ["area_sobre_suelo_sqft", "Area construida sobre el nivel del suelo"],
    ["area_sotano_sqft", "Area del sotano (0 si no tiene)"],
    ["anio_construccion", "Ano en que se construyo la vivienda"],
    ["anio_renovacion", "Ano de la ultima renovacion (vacio si nunca se renovo)"],
    ["direccion", "Direccion de la vivienda"],
    ["ciudad", "Ciudad donde esta ubicada"],
    ["estado", "Estado (siempre Washington, WA)"],
    ["codigo_postal", "Codigo postal de 5 digitos"],
    ["precio_por_sqft", "Precio dividido entre el area construida"],
    ["flag_precio_no_registrado", "Marca las ventas que llegaron con precio en 0"],
    ["flag_precio_extremo", "Marca los precios por pie2 que son imposibles"],
], columns=["Variable", "Que significa"])

assert set(variables_limpias["Variable"]).issubset(df.columns)
variables_limpias

,Variable,Que significa
0,id_inmueble,Identificador unico de cada vivienda
1,fecha,Dia en que se realizo la venta
2,precio,Precio de venta en dolares (variable objetivo ...
3,habitaciones,Numero de habitaciones
4,banos,Numero de banos
5,area_construida_sqft,"Area construida, en pies cuadrados"
6,area_lote_sqft,"Area del terreno, en pies cuadrados"
7,pisos,Numero de pisos
8,frente_agua,Si la vivienda tiene frente al agua (si/no)
9,vista,"Calidad de la vista, de 0 (sin vista) a 4 (exc..."


**Nota.** De estas 22, `precio` es la variable que queremos explicar (el Target); las dos ultimas (`flag_...`) no son caracteristicas de la vivienda sino controles de calidad que usamos para excluir ventas poco confiables de los analisis del precio. Todas las demas describen el tamano, la distribucion interna, el estado y la ubicacion de cada casa, y son la materia prima para las consultas SQL, los graficos y la evaluacion de variables que siguen.

## 5. Consultas SQL

Usamos SQL (con un motor llamado DuckDB que funciona directo en el notebook) para hacerle preguntas a los datos como si fueran una base de datos. De las muchas consultas posibles, elegimos tres que responden preguntas de negocio directas: que ciudades tienen los precios mas altos, si la antiguedad de la vivienda cambia la proporcion de ventas caras, y que tanto se venden algunas ciudades por encima del precio tipico de su propio codigo postal.

In [ ]:
import duckdb
con = duckdb.connect(":memory:")
con.register("ventas", df)